# Chapter 11: Pandas 

We saw in Chapter 9 how to import data from comma separated files (.csv) or other text files into numpy ndarrays with 1 or 2 axes, and how we can manipulate this data, add rows, columns etc.
One of the flaws of this approach is that it is really easy to make mistakes. While the data we read is structured (each row consisted of last name, first name, student id and several grades), we had to remember that midterm1 was a certain column, midterm2 a different one etc. These column numbers had no real meaning, and the order in which they were labelled was irrelevant. Basically, we have imported the *data* but lost the *meta-data*.

Maybe it would have been better to create a dictionary whose keys are the grade items and values list of grades:

In [87]:
import numpy as np
grades  = np.loadtxt('grades.csv',delimiter = ',', skiprows= 1, usecols=(3, 4, 5, 6, 7, 8, 9), max_rows=10)
# Create a dictionary of lists from this data:
keys = ("Quizzes average", "HW1 mark", "HW2 mark", "HW3 mark", "Exam1 mark", "Exam2 mark", "Final exam mark")
gradesD = {}
for col, k in enumerate(keys):
    print(col, k)
    gradesD[k] = grades[:,col]


0 Quizzes average
1 HW1 mark
2 HW2 mark
3 HW3 mark
4 Exam1 mark
5 Exam2 mark
6 Final exam mark


This makes operations on columns really easy:

In [88]:
gradesD['HW average'] = (gradesD['HW1 mark'] + gradesD['HW3 mark'] + gradesD['HW3 mark']) / 3

Operations on rows are a bit painful. Say we want all grades of Khan Vitaly with ID 7943944, we need to figure out that they are the 2nd students in the initial file

In [ ]:
row = 2
record = [gradesD[k][row] for k in gradesD.keys()]
print(record)

[np.float64(92.0), np.float64(79.0), np.float64(85.0), np.float64(92.0), np.float64(76.0), np.float64(60.0), np.float64(87.0), np.float64(87.66666666666667)]


In retrospect, maybe we should have created a list of dictionaries:


In [96]:
gradesL = []
for row in grades:
    record = {}
    for col, k in enumerate(keys):
        record[k] = row[col]
    gradesL.append(record)
print(gradesL)

[{'Quizzes average': np.float64(96.0), 'HW1 mark': np.float64(77.0), 'HW2 mark': np.float64(82.0), 'HW3 mark': np.float64(91.0), 'Exam1 mark': np.float64(88.0), 'Exam2 mark': np.float64(78.0), 'Final exam mark': np.float64(91.0)}, {'Quizzes average': np.float64(94.0), 'HW1 mark': np.float64(76.0), 'HW2 mark': np.float64(87.0), 'HW3 mark': np.float64(90.0), 'Exam1 mark': np.float64(90.0), 'Exam2 mark': np.float64(67.0), 'Final exam mark': np.float64(88.0)}, {'Quizzes average': np.float64(92.0), 'HW1 mark': np.float64(79.0), 'HW2 mark': np.float64(85.0), 'HW3 mark': np.float64(92.0), 'Exam1 mark': np.float64(76.0), 'Exam2 mark': np.float64(60.0), 'Final exam mark': np.float64(87.0)}, {'Quizzes average': np.float64(90.0), 'HW1 mark': np.float64(75.0), 'HW2 mark': np.float64(78.0), 'HW3 mark': np.float64(88.0), 'Exam1 mark': np.float64(90.0), 'Exam2 mark': np.float64(67.0), 'Final exam mark': np.float64(84.0)}, {'Quizzes average': np.float64(88.0), 'HW1 mark': np.float64(76.0), 'HW2 mark':

But now, operation on columns are difficult, and even looking for a student is painful...

The problem is that while dictionary are good to represent data with key:value structure, they are not really designed to represent data whose structure is more complex.

This is where Pandas comes into play (note that there are other approaches, including numpy structured arrays).

## 11.1 Pandas `Series`
*Series* are another type of container that can store data of various type. Series can be many things, single data (int, str, ...), lists, nparrays, or dictionaries.


In [100]:
import pandas as pd
rowL0 = pd.Series(grades[0])
rowL1 = pd.Series(grades[1])
print(rowL0)
print(rowL0[1])

0    96.0
1    77.0
2    82.0
3    91.0
4    88.0
5    78.0
6    91.0
dtype: float64
77.0


In [99]:
rowD0 = pd.Series(gradesL[0])
rowD1 = pd.Series(gradesL[1])
print(rowD0)
print(rowD0['HW1 mark'])

Quizzes average    96.0
HW1 mark           77.0
HW2 mark           82.0
HW3 mark           91.0
Exam1 mark         88.0
Exam2 mark         78.0
Final exam mark    91.0
dtype: float64
77.0


Series can be converted to dictionaries, lists, numpy arrays etc. Just like ndarras, operations on Series are vectorized:

In [ ]:
print(rowD0.to_list())

[620565656.0, 96.0, 77.0, 82.0, 91.0, 88.0, 78.0, 91.0]


In [63]:
print(rowL1.to_dict())

{0: 107856531.0, 1: 94.0, 2: 76.0, 3: 87.0, 4: 90.0, 5: 90.0, 6: 67.0, 7: 88.0}


In [66]:
print(rowL0 + rowL1)

0    728422187.0
1          190.0
2          153.0
3          169.0
4          181.0
5          178.0
6          145.0
7          179.0
dtype: float64


Pandas is actually pretty good at vectorizing operations, even when keys don't match exactly:

In [ ]:
rowD1['avg'] = 99
print(rowD0+rowD1)

Exam1 mark               178.0
Exam2 mark               145.0
Final exam mark          179.0
HW1 mark                 153.0
HW2 mark                 169.0
HW3 mark                 181.0
Quizzes average          190.0
Student number     728422187.0
avg                        NaN
dtype: float64


Finally, Series can be given a *name*. Here, maybe the student number could have been the name. Or maybe the students names could be the names. We'll see in a bit why. 

In [120]:
print(rowD0.name)
rowD0.name = '620565656'
rowD1.name = '107856531'
rowL0.name = '620565656'
rowL1.name = '107856531'
print(rowD0.name)
print(rowD0)

620565656
620565656
Quizzes average    96.0
HW1 mark           77.0
HW2 mark           82.0
HW3 mark           91.0
Exam1 mark         88.0
Exam2 mark         78.0
Final exam mark    91.0
Name: 620565656, dtype: float64


## 11.2 Pandas `DataFrame`

A `DataFrame` is a 2-dimensional labeled data structure with columns of potentially different types. One way to think about a `Dataframe` is as a two-dimensional list of data, where both rows and columns are structured, like a spreadsheet, or a `dict` of `Series`.
You can create a `DataFrame` from a list or dict of Series, a 2 axes ndarray

In [195]:
gradesDF = pd.DataFrame([rowD0, rowD1])
gradesDF

,Quizzes average,HW1 mark,HW2 mark,HW3 mark,Exam1 mark,Exam2 mark,Final exam mark
620565656,96.0,77.0,82.0,91.0,88.0,78.0,91.0
107856531,94.0,76.0,87.0,90.0,90.0,67.0,88.0


In [197]:
print(gradesDF['HW1 mark'])
print(gradesDF.loc['620565656'])

620565656    77.0
107856531    76.0
Name: HW1 mark, dtype: float64
Quizzes average    96.0
HW1 mark           77.0
HW2 mark           82.0
HW3 mark           91.0
Exam1 mark         88.0
Exam2 mark         78.0
Final exam mark    91.0
Name: 620565656, dtype: float64


Of course, pandas knows how to read a file directly into a DataFrame.

In [198]:
gradesDF2 = pd.read_csv("grades.csv", skipinitialspace=True, index_col=2)
gradesDF2.head() # same as gradesDF2[:5]

,#First name,Last name,Quizzes average,HW1 mark,HW2 mark,HW3 mark,Exam1 mark,Exam2 mark,Final exam mark
Student number,,,,,,,,,
620565656,BARRY,LAMB,96,77,82,91,88,78,91
107856531,NARIMAN,PELUCHONDELAGARZA,94,76,87,90,90,67,88
7943944,VITALY,KHAN,92,79,85,92,76,60,87
114037001,NATALYA,SUTRADHAR,90,75,78,88,90,67,84
581943811,MAGARET,HIRJI,88,76,91,98,96,56,81


We now have the best of a list of dictionaries and a dictionary of lists:

In [207]:
print(type(gradesDF2['HW avg']))
print(type(gradesDF2.loc[7943944]))

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>


In [ ]:
# selecting rows:

# loc is label-based, iloc is index-based
print(gradesDF2.loc[114037001])
print(gradesDF2.iloc[2])


#First name          NATALYA
Last name          SUTRADHAR
Quizzes average           90
HW1 mark                  75
HW2 mark                  78
HW3 mark                  88
Exam1 mark                90
Exam2 mark                67
Final exam mark           84
Name: 114037001, dtype: object
#First name        VITALY
Last name            KHAN
Quizzes average        92
HW1 mark               79
HW2 mark               85
HW3 mark               92
Exam1 mark             76
Exam2 mark             60
Final exam mark        87
Name: 7943944, dtype: object


In [ ]:
# selecting columns
print(gradesDF2['HW1 mark'][:10])


Student number
620565656    77
107856531    76
7943944      79
114037001    75
581943811    76
367541502    70
602646616    81
282046779    75
961460380    75
527925346    70
Name: HW1 mark, dtype: int64


In [200]:
# Operation on columns
gradesDF2['HW avg'] = (gradesDF2['HW1 mark'] + gradesDF2['HW2 mark'] + gradesDF2['HW3 mark']) / 3
gradesDF2.head()

,#First name,Last name,Quizzes average,HW1 mark,HW2 mark,HW3 mark,Exam1 mark,Exam2 mark,Final exam mark,HW avg
Student number,,,,,,,,,,
620565656,BARRY,LAMB,96,77,82,91,88,78,91,83.333333
107856531,NARIMAN,PELUCHONDELAGARZA,94,76,87,90,90,67,88,84.333333
7943944,VITALY,KHAN,92,79,85,92,76,60,87,85.333333
114037001,NATALYA,SUTRADHAR,90,75,78,88,90,67,84,80.333333
581943811,MAGARET,HIRJI,88,76,91,98,96,56,81,88.333333


## 11.3 Sorting

In [232]:
gradesDF2.sort_values('HW avg', ascending=False).head()

,#First name,Last name,Quizzes average,HW1 mark,HW2 mark,HW3 mark,Exam1 mark,Exam2 mark,Final exam mark,HW avg
Student number,,,,,,,,,,
441797054,ZHONG,TUNCAY,100,87,93,91,87,74,90,90.333333
882363866,KOHL,SU,94,82,98,91,91,74,90,90.333333
463318745,BROOKLYN,SCOTT,98,83,94,91,91,66,90,89.333333
586057811,LORI-ANN,VANOIRSCHOT,92,84,86,98,91,64,82,89.333333
128992445,MARITZA,YIN,93,82,98,87,90,64,97,89.000000


## 11.4 Selecting / filtering

In [259]:
A = gradesDF2['HW avg'] >= 90
gradesDF2[A]

,#First name,Last name,Quizzes average,HW1 mark,HW2 mark,HW3 mark,Exam1 mark,Exam2 mark,Final exam mark,HW avg
Student number,,,,,,,,,,
882363866,KOHL,SU,94,82,98,91,91,74,90,90.333333
441797054,ZHONG,TUNCAY,100,87,93,91,87,74,90,90.333333


In [265]:
B = (gradesDF2['HW avg'] >= 85) & (gradesDF2['HW avg'] < 90)
gradesDF2[B]

,#First name,Last name,Quizzes average,HW1 mark,HW2 mark,HW3 mark,Exam1 mark,Exam2 mark,Final exam mark,HW avg
Student number,,,,,,,,,,
7943944,VITALY,KHAN,92,79,85,92,76,60,87,85.333333
581943811,MAGARET,HIRJI,88,76,91,98,96,56,81,88.333333
602646616,ERICKA,HOU,93,81,90,86,90,76,92,85.666667
282046779,ABDOOL,VILLASPIN,87,75,90,90,85,70,89,85.000000
961460380,FERN,HUANG,100,75,91,91,97,53,91,85.666667
...,...,...,...,...,...,...,...,...,...,...
894355623,ANKUSH,GRAHAM,97,76,83,99,92,76,81,86.000000
485307898,SUKHPREET,PASPALOFSKI,93,78,90,89,80,69,91,85.666667
764383512,DAMIEN,JEGAJEEVAN,92,69,95,94,81,65,76,86.000000


In [ ]:
# Another way using a 'query'. Note the back quotes `
B2 = gradesDF2.query("80 <= `HW avg` < 90")


,#First name,Last name,Quizzes average,HW1 mark,HW2 mark,HW3 mark,Exam1 mark,Exam2 mark,Final exam mark,HW avg
Student number,,,,,,,,,,
620565656,BARRY,LAMB,96,77,82,91,88,78,91,83.333333
107856531,NARIMAN,PELUCHONDELAGARZA,94,76,87,90,90,67,88,84.333333
7943944,VITALY,KHAN,92,79,85,92,76,60,87,85.333333
114037001,NATALYA,SUTRADHAR,90,75,78,88,90,67,84,80.333333
581943811,MAGARET,HIRJI,88,76,91,98,96,56,81,88.333333
...,...,...,...,...,...,...,...,...,...,...
147970372,ROXANN,GARNER,93,72,87,87,91,72,91,82.000000
248015969,JANAYA,ELZOEIBY,94,72,82,91,92,69,93,81.666667
970187185,PATRYK,BERNIQUER,95,72,88,94,88,67,95,84.666667
